In [ ]:
import os
os.listdir('Jupiter/')

In [ ]:
import glob

In [ ]:
from astropy.io import fits

In [ ]:
for fn in glob.glob('Jupiter/*FIT'):
    header = fits.getheader(fn)
    print(fn, header['FILTER'], header['EXPTIME'], header['CCD-TEMP'])

In [ ]:
for fn in glob.glob('Darks/*40ms*FIT'):
    header = fits.getheader(fn)
    print(fn, header['FILTER'], header['EXPTIME'], header['CCD-TEMP'])

In [ ]:
ls "Darks"

In [ ]:
ls "Twilight Flats"

In [ ]:
for fn in glob.glob('Twilight Flats/*FIT'):
    data = fits.getdata(fn)
    print(f"File={fn:40s}, {np.min(data):10d}, {np.mean(data):10.1f}, {np.median(data):10.1f}, {np.max(data):10.1f}")

In [ ]:
combined_dark = np.median([fits.getdata(fn) for fn in glob.glob('Darks/*40ms*.FIT')], axis=0)

In [ ]:
file_list = glob.glob('Darks/*40ms*FIT')
timeseries_of_darks = np.array([fits.getdata(fn) for fn in file_list])
stddev_of_dark_timeseries = timeseries_of_darks.std(axis=0)
readnoise_estimate = np.mean(stddev_of_dark_timeseries)
readnoise_estimate

In [ ]:
%matplotlib inline
import pylab as pl
pl.style.use('dark_background')
pl.rcParams['image.origin'] = 'lower'

In [ ]:
from astropy import visualization

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(combined_dark, norm=visualization.simple_norm(combined_dark, stretch='asinh'))
pl.colorbar()

In [ ]:
file_list = glob.glob('Twilight Flats/flat_10s_I00*.FIT')
timeseries_of_IbandFlats = np.array([fits.getdata(fn) for fn in file_list])

In [ ]:
timeseries_of_IbandFlats.shape

In [ ]:
timeseries_of_IbandFlats.mean(axis=(1,2))

In [ ]:
dark_subtracted_Iband_flats = timeseries_of_IbandFlats - combined_dark
dark_subtracted_combined_Iband_flat = dark_subtracted_Iband_flats.mean(axis=0)
normed_Iband_flat = dark_subtracted_combined_Iband_flat / dark_subtracted_combined_Iband_flat.mean()

In [ ]:
visualization.simple_norm?

In [ ]:
510 * 765 * 0.005

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(normed_Iband_flat, norm=visualization.simple_norm(normed_Iband_flat,
                                                                              stretch='linear',
                                                                              min_percent=0.5,
                                                                              max_percent=99.5))
pl.colorbar()

In [ ]:
file_list = glob.glob('Jupiter\jupiter_40ms_I00*.FIT')
timeseries_of_JupiterIBand = np.array([fits.getdata(fn) for fn in file_list])

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(timeseries_of_JupiterIBand[0], norm=visualization.simple_norm(timeseries_of_JupiterIBand[0],
                                                                              stretch='log',
                                                                              min_percent=0.5,
                                                                              max_percent=100))
pl.colorbar()

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(timeseries_of_JupiterIBand[0], norm=visualization.simple_norm(timeseries_of_JupiterIBand[0],
                                                                              stretch='linear',
                                                                              min_percent=97,
                                                                              max_percent=100))
pl.colorbar()

In [ ]:
darksubtracted_timeseries_of_JupiterIBand = timeseries_of_JupiterIBand - combined_dark
darksubtracted_timeseries_of_JupiterIBand.shape

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(darksubtracted_timeseries_of_JupiterIBand[0], norm=visualization.simple_norm(darksubtracted_timeseries_of_JupiterIBand[0],
                                                                              stretch='log',
                                                                              min_percent=0.5,
                                                                              max_percent=100))
pl.colorbar()

# Fix the I-band offsets

In [ ]:
darksubtracted_flatfielded_timeseries_of_JupiterIBand = darksubtracted_timeseries_of_JupiterIBand / normed_Iband_flat

refY,refX = np.unravel_index(np.argmax(darksubtracted_flatfielded_timeseries_of_JupiterIBand[0][Imoonslc]),
                 darksubtracted_flatfielded_timeseries_of_JupiterIBand[0][Imoonslc].shape)
refY,refX

In [ ]:
for img in darksubtracted_flatfielded_timeseries_of_JupiterIBand:
    pkY,pkX = np.unravel_index(np.argmax(img[Imoonslc]),
                 img[Imoonslc].shape)
    print(f"Shift={refY-pkY}, {refX-pkX}")

In [ ]:
darksubtracted_flatfielded_timeseries_of_JupiterIBand = darksubtracted_timeseries_of_JupiterIBand / normed_Iband_flat


darksubtracted_flatfielded_timeseries_of_JupiterIBand[2,:,:] = np.roll(darksubtracted_flatfielded_timeseries_of_JupiterIBand[2,:,:], -2, axis=0)
darksubtracted_flatfielded_timeseries_of_JupiterIBand[2,:,:] = np.roll(darksubtracted_flatfielded_timeseries_of_JupiterIBand[2,:,:], 1, axis=1)
darksubtracted_flatfielded_timeseries_of_JupiterIBand[3,:,:] = np.roll(darksubtracted_flatfielded_timeseries_of_JupiterIBand[3,:,:], -3, axis=0)
darksubtracted_flatfielded_timeseries_of_JupiterIBand[3,:,:] = np.roll(darksubtracted_flatfielded_timeseries_of_JupiterIBand[3,:,:], 2, axis=1)
darksubtracted_flatfielded_timeseries_of_JupiterIBand[4,:,:] = np.roll(darksubtracted_flatfielded_timeseries_of_JupiterIBand[4,:,:], -2, axis=0)
darksubtracted_flatfielded_timeseries_of_JupiterIBand[4,:,:] = np.roll(darksubtracted_flatfielded_timeseries_of_JupiterIBand[4,:,:], 1, axis=1)

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(darksubtracted_flatfielded_timeseries_of_JupiterIBand[0],
          norm=visualization.simple_norm(darksubtracted_flatfielded_timeseries_of_JupiterIBand[0],
                                                                              stretch='log',
                                                                              min_percent=0.5,
                                                                              max_percent=100))
pl.colorbar()

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(darksubtracted_flatfielded_timeseries_of_JupiterIBand[0][250:370,200:320],
          norm=visualization.simple_norm(darksubtracted_flatfielded_timeseries_of_JupiterIBand[0][250:370,200:320],
                                                                              stretch='linear',
                                                                              min_percent=0.5,
                                                                              max_percent=100))
pl.colorbar()

In [ ]:
darksubtracted_flatfielded_timeseries_of_JupiterIBand.shape

In [ ]:
pl.figure(figsize=(12,7))
for ii,image in enumerate(darksubtracted_flatfielded_timeseries_of_JupiterIBand):
    ax = pl.subplot(1,5,ii+1)
    ax.imshow(image[250:370,200:320],
              norm=visualization.simple_norm(image[250:370,200:320],
                                             stretch='linear',
                                             min_percent=0.5,
                                             max_percent=100))

In [ ]:
pl.figure(figsize=(12,7))
for ii,image in enumerate(darksubtracted_flatfielded_timeseries_of_JupiterIBand):
    ax = pl.subplot(1,5,ii+1)
    Imoonslc = slc = slice(150, 170), slice(365,380)
    ax.imshow(image[slc],
              norm=visualization.simple_norm(image[slc],
                                             stretch='linear',
                                             min_percent=0.5,
                                             max_percent=100))

In [ ]:
Iband_Jupiter_median = np.median(darksubtracted_flatfielded_timeseries_of_JupiterIBand, axis=0)

In [ ]:
pl.figure(figsize=(12,7))
Islc = slice(250,370), slice(200,320)
pl.imshow(Iband_Jupiter_median[Islc],
          norm=visualization.simple_norm(Iband_Jupiter_median[Islc],
                                                                              stretch='linear',
                                                                              min_percent=0.5,
                                                                              max_percent=100))
pl.colorbar()
pl.title("I-band Combined Image of Jupiter")

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(Iband_Jupiter_median[slc],
          norm=visualization.simple_norm(Iband_Jupiter_median[slc],
                                                                              stretch='linear',
                                                                              min_percent=0.5,
                                                                              max_percent=100))
pl.colorbar()

In [ ]:
readnoise_estimate

$$D_{comb} = \frac{\Sigma_i^N D_{i}}{N}$$


$$\sigma_{sum}^2 = \Sigma \sigma_i^2$$

$$\mu = \Sigma / N$$

$$\sigma_\mu^2 = \sigma_i^2 \ N^2$$

$$\sigma^2_{D,comb} = \frac{ \Sigma_{i}^N \sigma_{i}^2 } { N^2 }$$

In [ ]:
combined_dark_uncertainty = (stddev_of_dark_timeseries**2 * 5 / 5**2)**0.5

In [ ]:
combined_dark_uncertainty = stddev_of_dark_timeseries / np.sqrt(len(timeseries_of_darks))

# Uncertainty on the flats

$$\mu_{flat} = \frac{\Sigma F_i}{N}$$

How do we estimate $\sigma_{F}$? 

$F = F_{raw} - D$

$\sigma_{F}^2 = \sigma_{F,raw}^2 + \sigma_D^2$

$F$ is a number of photons - it is our best estimate of the photon counts.

Therefore, we can use Poisson statistics:
$\sigma_F \approx \sqrt{F}$

In [ ]:
photon_noise_Iband_flat = dark_subtracted_combined_Iband_flat**0.5

In [ ]:
total_noise_Iband_flat = (photon_noise_Iband_flat**2 +
                          combined_dark_uncertainty**2 +
                          readnoise_estimate**2)**0.5
average_Iband_flat_uncertainty = total_noise_Iband_flat / np.sqrt(len(timeseries_of_IbandFlats))

$$ z = a x $$
$$\sigma^2_z = a^2 \sigma_x^2$$

When we normalized, we multiplied our image by a constant $a$, which in this case was just the mean

In [ ]:
norm_Iband_flat_uncertainty = average_Iband_flat_uncertainty / dark_subtracted_combined_Iband_flat.mean()

In [ ]:
norm_Iband_flat_uncertainty

# Uncertainty on Jupiter images

$\sigma_{science}^2 = \sigma_{photon}^2 + \sigma_{readnoise}^2 +ish \sigma_{flat}^2$

In [ ]:
photon_noise_estimate_jupiter_Iband = darksubtracted_timeseries_of_JupiterIBand**0.5

In [ ]:
photon_noise_estimate_jupiter_Iband[np.isnan(photon_noise_estimate_jupiter_Iband)] = 0

$$z = x/y$$
$$\frac{\sigma_z^2}{z^2} = \frac{\sigma_x^2}{x^2} + \frac{\sigma_y^2}{y^2}$$

In [ ]:
total_noise_estimate_jupiter_Iband = (photon_noise_estimate_jupiter_Iband**2 + 
                                      combined_dark_uncertainty**2 +
                                      readnoise_estimate**2)**0.5

In [ ]:
total_flat_noise_estimate_Jupiter_Iband = (((
    total_noise_estimate_jupiter_Iband**2 /
    darksubtracted_timeseries_of_JupiterIBand**2)
    + (norm_Iband_flat_uncertainty**2 / normed_Iband_flat**2)) * 
    darksubtracted_flatfielded_timeseries_of_JupiterIBand**2)**0.5

In [ ]:
total_flat_noise_estimate_Jupiter_Iband.shape

$$ \mu = \frac{\Sigma_i x_i}{N}$$
$z=\mu$

$$ \sigma_z^2 = \frac{\Sigma_i \sigma_{x,i}^2}{N^2}$$

In [ ]:
final_mean_Jupiter_IBand_noise = ((
    total_flat_noise_estimate_Jupiter_Iband**2).sum(axis=0) /
    total_flat_noise_estimate_Jupiter_Iband.shape[0]**2)**0.5

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(final_mean_Jupiter_IBand_noise[250:370,200:320],
          norm=visualization.simple_norm(final_mean_Jupiter_IBand_noise[250:370,200:320],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.title("Uncertainty on I-band Combined Image of Jupiter")

# B band

In [ ]:
file_list = glob.glob('Twilight Flats/flat_15s_B00*.FIT')
timeseries_of_BbandFlats = np.array([fits.getdata(fn) for fn in file_list])
dark_subtracted_Bband_flats = timeseries_of_BbandFlats - combined_dark
dark_subtracted_combined_Bband_flat = dark_subtracted_Bband_flats.mean(axis=0)
normed_Bband_flat = dark_subtracted_combined_Bband_flat / dark_subtracted_combined_Bband_flat.mean()

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(normed_Bband_flat, norm=visualization.simple_norm(normed_Bband_flat,
                                                            stretch='linear',
                                                            min_percent=0.5,
                                                            max_percent=99.5))
pl.colorbar()

In [ ]:
file_list = glob.glob('Jupiter\jupiter_40ms_B00*.FIT')
timeseries_of_JupiterBband = np.array([fits.getdata(fn) for fn in file_list])
darksubtracted_timeseries_of_JupiterBband = timeseries_of_JupiterBband - combined_dark
darksubtracted_flatfielded_timeseries_of_JupiterBband = darksubtracted_timeseries_of_JupiterBband / normed_Bband_flat
Bband_Jupiter_median = np.median(darksubtracted_flatfielded_timeseries_of_JupiterBband, axis=0)

In [ ]:
pl.figure(figsize=(12,7))
for ii,image in enumerate(darksubtracted_flatfielded_timeseries_of_JupiterBband):
    ax = pl.subplot(1,5,ii+1)
    bslc = slice(270,390), slice(200,320)
    ax.imshow(image[bslc],
              norm=visualization.simple_norm(image[bslc],
                                             stretch='linear',
                                             min_percent=0.5,
                                             max_percent=100))

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(Bband_Jupiter_median[bslc],
          norm=visualization.simple_norm(Bband_Jupiter_median[bslc],
                                                                              stretch='linear',
                                                                              min_percent=0.5,
                                                                              max_percent=100))
pl.colorbar()
pl.title("B-band Combined Image of Jupiter")

In [ ]:
photon_noise_Bband_flat = dark_subtracted_combined_Bband_flat**0.5
total_noise_Bband_flat = (photon_noise_Bband_flat**2 +
                          combined_dark_uncertainty**2 +
                          readnoise_estimate**2)**0.5
average_Bband_flat_uncertainty = total_noise_Bband_flat / np.sqrt(len(timeseries_of_BbandFlats))
norm_Bband_flat_uncertainty = average_Bband_flat_uncertainty / dark_subtracted_combined_Bband_flat.mean()

In [ ]:
photon_noise_estimate_jupiter_Bband = darksubtracted_timeseries_of_JupiterBband**0.5
photon_noise_estimate_jupiter_Bband[np.isnan(photon_noise_estimate_jupiter_Bband)] = 0
total_noise_estimate_jupiter_Bband = (photon_noise_estimate_jupiter_Bband**2 + 
                                      combined_dark_uncertainty**2 +
                                      readnoise_estimate**2)**0.5
total_flat_noise_estimate_Jupiter_Bband = (((
    total_noise_estimate_jupiter_Bband**2 /
    darksubtracted_timeseries_of_JupiterBband**2)
    + (norm_Bband_flat_uncertainty**2 / normed_Bband_flat**2)) * 
    darksubtracted_flatfielded_timeseries_of_JupiterBband**2)**0.5
final_mean_Jupiter_Bband_noise = ((
    total_flat_noise_estimate_Jupiter_Bband**2).sum(axis=0) /
    total_flat_noise_estimate_Jupiter_Bband.shape[0]**2)**0.5

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(final_mean_Jupiter_Bband_noise[bslc],
          norm=visualization.simple_norm(final_mean_Jupiter_Bband_noise[bslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.title("Uncertainty on B-band Combined Image of Jupiter")

# V-band

In [ ]:
file_list = glob.glob('Twilight Flats/flat_15s_V00*.FIT')
timeseries_of_VbandFlats = np.array([fits.getdata(fn) for fn in file_list])
dark_subtracted_Vband_flats = timeseries_of_VbandFlats - combined_dark
dark_subtracted_combined_Vband_flat = dark_subtracted_Vband_flats.mean(axis=0)
normed_Vband_flat = dark_subtracted_combined_Vband_flat / dark_subtracted_combined_Vband_flat.mean()

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(normed_Vband_flat, norm=visualization.simple_norm(normed_Vband_flat,
                                                            stretch='linear',
                                                            min_percent=0.5,
                                                            max_percent=99.5))
pl.colorbar()

In [ ]:
file_list = glob.glob('Jupiter\jupiter_40ms_V00*.FIT')
timeseries_of_JupiterVband = np.array([fits.getdata(fn) for fn in file_list])
darksubtracted_timeseries_of_JupiterVband = timeseries_of_JupiterVband - combined_dark
darksubtracted_flatfielded_timeseries_of_JupiterVband = darksubtracted_timeseries_of_JupiterVband / normed_Vband_flat
Vband_Jupiter_median = np.median(darksubtracted_flatfielded_timeseries_of_JupiterVband, axis=0)

In [ ]:
pl.figure(figsize=(12,7))
for ii,image in enumerate(darksubtracted_flatfielded_timeseries_of_JupiterVband):
    ax = pl.subplot(1,5,ii+1)
    vslc = bslc
    ax.imshow(image[bslc],
              norm=visualization.simple_norm(image[bslc],
                                             stretch='linear',
                                             min_percent=0.5,
                                             max_percent=100))

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(Vband_Jupiter_median[vslc],
          norm=visualization.simple_norm(Vband_Jupiter_median[vslc],
                                                                              stretch='linear',
                                                                              min_percent=0.5,
                                                                              max_percent=100))
pl.colorbar()
pl.title("V-band Combined Image of Jupiter")

In [ ]:
photon_noise_Vband_flat = dark_subtracted_combined_Vband_flat**0.5
total_noise_Vband_flat = (photon_noise_Vband_flat**2 +
                          combined_dark_uncertainty**2 +
                          readnoise_estimate**2)**0.5
average_Vband_flat_uncertainty = total_noise_Vband_flat / np.sqrt(len(timeseries_of_VbandFlats))
norm_Vband_flat_uncertainty = average_Vband_flat_uncertainty / dark_subtracted_combined_Vband_flat.mean()

In [ ]:
photon_noise_estimate_jupiter_Vband = darksubtracted_timeseries_of_JupiterVband**0.5
photon_noise_estimate_jupiter_Vband[np.isnan(photon_noise_estimate_jupiter_Vband)] = 0
total_noise_estimate_jupiter_Vband = (photon_noise_estimate_jupiter_Vband**2 + 
                                      combined_dark_uncertainty**2 +
                                      readnoise_estimate**2)**0.5
total_flat_noise_estimate_Jupiter_Vband = (((
    total_noise_estimate_jupiter_Vband**2 /
    darksubtracted_timeseries_of_JupiterVband**2)
    + (norm_Vband_flat_uncertainty**2 / normed_Vband_flat**2)) * 
    darksubtracted_flatfielded_timeseries_of_JupiterVband**2)**0.5
final_mean_Jupiter_Vband_noise = ((
    total_flat_noise_estimate_Jupiter_Vband**2).sum(axis=0) /
    total_flat_noise_estimate_Jupiter_Vband.shape[0]**2)**0.5

In [ ]:
pl.figure(figsize=(12,7))
pl.imshow(final_mean_Jupiter_Vband_noise[vslc],
          norm=visualization.simple_norm(final_mean_Jupiter_Vband_noise[vslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.title("Uncertainty on V-band Combined Image of Jupiter")

In [ ]:
SNR_I = Iband_Jupiter_median / final_mean_Jupiter_IBand_noise
SNR_B = Bband_Jupiter_median / final_mean_Jupiter_Bband_noise
SNR_V = Vband_Jupiter_median / final_mean_Jupiter_Vband_noise

In [ ]:
pl.figure(figsize=(15,4))
pl.subplot(1,3,1)
pl.imshow(SNR_I[Islc],
          norm=visualization.simple_norm(SNR_I[Islc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.title("SNR_I")
pl.subplot(1,3,2)
pl.imshow(SNR_B[bslc],
          norm=visualization.simple_norm(SNR_B[bslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.title("SNR_B")
pl.subplot(1,3,3)
pl.imshow(SNR_V[vslc],
          norm=visualization.simple_norm(SNR_V[vslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.title("SNR_V")

In [ ]:
pl.figure(figsize=(15,4))
pl.subplot(1,3,1)
pl.imshow(Iband_Jupiter_median[Islc],
          norm=visualization.simple_norm(Iband_Jupiter_median[Islc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.title("Jupiter I")
pl.subplot(1,3,2)
pl.imshow(Bband_Jupiter_median[bslc],
          norm=visualization.simple_norm(Bband_Jupiter_median[bslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.title("Jupiter B")
pl.subplot(1,3,3)
pl.imshow(Vband_Jupiter_median[vslc],
          norm=visualization.simple_norm(Vband_Jupiter_median[vslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.title("Jupiter V")

In [ ]:
total_counts_jupiter_I = Iband_Jupiter_median[Islc].sum()
total_counts_jupiter_V = Vband_Jupiter_median[vslc].sum()
total_counts_jupiter_B = Bband_Jupiter_median[bslc].sum()
total_counts_jupiter_I, total_counts_jupiter_V, total_counts_jupiter_B

In [ ]:
exptime_I = fits.getheader('Jupiter\jupiter_40ms_I001.FIT')['EXPTIME']
exptime_B = fits.getheader('Jupiter\jupiter_40ms_B001.FIT')['EXPTIME']
exptime_V = fits.getheader('Jupiter\jupiter_40ms_V001.FIT')['EXPTIME']

exptime_I, exptime_B, exptime_V

In [ ]:
rate_I = total_counts_jupiter_I / exptime_I
rate_V = total_counts_jupiter_V / exptime_V
rate_B = total_counts_jupiter_B / exptime_B
print(f"I={rate_I:0.3g} ph/s, B={rate_B:0.3g} ph/s, V={rate_V:0.3g} ph/s")

$$z = x +y $$
$$ \sigma_z^2 = \sigma_x^2 + \sigma_y^2$$

$$\sigma_{sum} = \left( \Sigma \sigma_i^2 \right)^{1/2}$$

In [ ]:
uncertainty_cts_I = np.nansum(final_mean_Jupiter_IBand_noise[Islc]**2)**0.5
uncertainty_cts_V = np.nansum(final_mean_Jupiter_Vband_noise[vslc]**2)**0.5
uncertainty_cts_B = np.nansum(final_mean_Jupiter_Bband_noise[bslc]**2)**0.5
uncertainty_cts_I, uncertainty_cts_V, uncertainty_cts_B

In [ ]:
uncertainty_rate_I = uncertainty_cts_I / exptime_I
uncertainty_rate_V = uncertainty_cts_V / exptime_V
uncertainty_rate_B = uncertainty_cts_B / exptime_B

In [ ]:
print(f"I={rate_I:12.3g} +/- {uncertainty_rate_I:12.3g} ph/s (SNR={rate_I/uncertainty_rate_I:10.3g})")
print(f"V={rate_V:12.3g} +/- {uncertainty_rate_V:12.3g} ph/s (SNR={rate_V/uncertainty_rate_V:10.3g})")
print(f"B={rate_B:12.3g} +/- {uncertainty_rate_B:12.3g} ph/s (SNR={rate_B/uncertainty_rate_B:10.3g})")

# Color Image

In [ ]:
norm_I = visualization.simple_norm(Iband_Jupiter_median[Islc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100)
norm_V = visualization.simple_norm(Vband_Jupiter_median[vslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100)
norm_B = visualization.simple_norm(Vband_Jupiter_median[bslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100)



jupiter_color = np.array([norm_I(Iband_Jupiter_median),
                          norm_V(Vband_Jupiter_median),
                          norm_B(Bband_Jupiter_median)])
jupiter_color = jupiter_color.T.swapaxes(0,1)
jupiter_color[:,:,0] = np.roll(jupiter_color[:,:,0], 22, axis=0)
jupiter_color[:,:,0] = np.roll(jupiter_color[:,:,0], -3, axis=1)

In [ ]:
Bslc3d = (bslc[0], bslc[1], slice(None))

In [ ]:
pl.imshow(jupiter_color[Bslc3d])

In [ ]:
moonbslc3d = slice(170, 190), slice(360,380), slice(None)

In [ ]:
pl.imshow(jupiter_color[moonbslc3d])

In [ ]:
moonslc = slice(150, 200), slice(360,380)
pl.figure(figsize=(12,4))
pl.subplot(1,3,1)
pl.imshow(Iband_Jupiter_median[moonslc],
          norm=visualization.simple_norm(Iband_Jupiter_median[moonslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.subplot(1,3,2)
pl.imshow(Vband_Jupiter_median[moonslc],
          norm=visualization.simple_norm(Vband_Jupiter_median[moonslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()
pl.subplot(1,3,3)
pl.imshow(Bband_Jupiter_median[moonslc],
          norm=visualization.simple_norm(Bband_Jupiter_median[moonslc],
                                         stretch='linear',
                                         min_percent=0.5,
                                         max_percent=100))
pl.colorbar()

In [ ]:
refY,refX = np.unravel_index(np.argmax(Vband_Jupiter_median[moonslc]),
                 Vband_Jupiter_median[moonslc].shape)
refY,refX

In [ ]:
I_Y,I_X = np.unravel_index(np.argmax(Iband_Jupiter_median[moonslc]),
                 Iband_Jupiter_median[moonslc].shape)
B_Y,B_X = np.unravel_index(np.argmax(Bband_Jupiter_median[moonslc]),
                 Bband_Jupiter_median[moonslc].shape)

In [ ]:
B_X-refX, B_Y-refY

In [ ]:
I_X-refX, I_Y-refY

In [ ]:
jupiter_color = np.array([norm_I(Iband_Jupiter_median),
                          norm_V(Vband_Jupiter_median),
                          norm_B(Bband_Jupiter_median)])
jupiter_color = jupiter_color.T.swapaxes(0,1)
jupiter_color[:,:,0] = np.roll(jupiter_color[:,:,0], 24, axis=0)
jupiter_color[:,:,0] = np.roll(jupiter_color[:,:,0], -4, axis=1)
jupiter_color[:,:,2] = np.roll(jupiter_color[:,:,2], -5, axis=0)
jupiter_color[:,:,2] = np.roll(jupiter_color[:,:,2], -3, axis=1)

In [ ]:
pl.imshow(jupiter_color[Bslc3d])

In [ ]:
pl.imshow(jupiter_color[moonbslc3d])

In [ ]:
pwd

In [ ]:
fits.PrimaryHDU(data=jupiter_color.T.swapaxes(0,1)).writeto("Jupiter_Color.fits",
                                                           overwrite=True)

In [ ]:
import PIL

In [ ]:
jupiter_color_png = (jupiter_color*255).astype('uint8')

In [ ]:
pl.imshow(jupiter_color_png[Bslc3d])

In [ ]:
PIL.Image.fromarray(jupiter_color_png[::-1,:,:]).save('jupiter_color.png')

In [ ]:
PIL.Image.fromarray(jupiter_color_png)

In [ ]:
np.array(PIL.Image.open('jupiter_test.png')).dtype

In [ ]:
from IPython.display import Image

In [ ]:
Image('jupiter_color.png')

In [ ]:
pl.plot((jupiter_color[Bslc3d][:,:,0] / jupiter_color[Bslc3d][:,:,1]).ravel(),
        (jupiter_color[Bslc3d][:,:,1] / jupiter_color[Bslc3d][:,:,2]).ravel(),
       ',')
pl.axis([0,4,0,15])

In [ ]:
pl.imshow(jupiter_color[Bslc3d])

In [ ]:
pl.imshow((jupiter_color[Bslc3d][:,:,0] / jupiter_color[Bslc3d][:,:,1]) > 1.05)